In [2]:

# Candidate-Elimination Algorithm
# Machine Learning Lab Program

import pandas as pd
import io

# Step 1: Create a sample CSV dataset
csv_data = """Sky,AirTemp,Humidity,Wind,Water,Forecast,EnjoySport
Sunny,Warm,Normal,Strong,Warm,Same,Yes
Sunny,Warm,High,Strong,Warm,Same,Yes
Rainy,Cold,High,Strong,Warm,Change,No
Sunny,Warm,High,Strong,Cool,Change,Yes
"""

# Read the CSV file into a DataFrame
df = pd.read_csv(io.StringIO(csv_data))

print("Training Dataset:")
print(df.to_string(index=False))

# Step 2: Separate attributes and target
X = df.iloc[:, :-1].values.tolist()
Y = df.iloc[:, -1].tolist()

num_attributes = len(X[0])

# Step 3: Initialize S and G boundaries
S = ['Ø'] * num_attributes
G = [['?'] * num_attributes]

def is_consistent(h, x):
    return all(h[i] == '?' or h[i] == x[i]
               for i in range(len(h)))

def more_general_or_equal(h1, h2):
    return all(a == '?' or a == b
               for a, b in zip(h1, h2))

def generalize_S(s, x):
    new_s = s.copy()
    for i in range(len(s)):
        if s[i] == 'Ø':
            new_s[i] = x[i]
        elif s[i] != x[i]:
            new_s[i] = '?'
    return new_s

def specialize_G(g, x, domains):
    specializations = []
    for i in range(len(g)):
        if g[i] == '?':
            for value in domains[i]:
                if value != x[i]:
                    new_g = g.copy()
                    new_g[i] = value
                    specializations.append(new_g)
    return specializations

# Get possible values of each attribute
domains = [
    sorted(df[col].unique().tolist())
    for col in df.columns[:-1]
]

# Step 4: Process each training example
for index, (x, label) in enumerate(zip(X, Y)):
    print(f"\nExample {index + 1}: {x} -> {label}")

    if label == 'Yes':
        # Remove hypotheses from G that do not cover positive example
        G = [g for g in G if is_consistent(g, x)]

        # Generalize S to cover the positive example
        S = generalize_S(S, x)

        # Keep S maximally specific
        G = [g for g in G
             if more_general_or_equal(g, S)]

    else:
        # Remove hypotheses from S that cover negative example
        if is_consistent(S, x):
            # A negative example covered by S may indicate
            # that no consistent hypothesis remains in this boundary.
            S = ['Ø'] * num_attributes
            print("Warning: S requires reconsideration.")

        # Specialize G to exclude the negative example
        new_G = []
        for g in G:
            if is_consistent(g, x):
                for h in specialize_G(g, x, domains):
                    if more_general_or_equal(h, S):
                        new_G.append(h)
            else:
                new_G.append(g)

        # Remove duplicate hypotheses
        G = [list(h) for h in {tuple(h) for h in new_G}]

    print("Specific Boundary S:", S)
    print("General Boundary G:", G)

# Step 5: Display final boundaries
print("\nFinal Specific Boundary (S):")
print(S)

print("\nFinal General Boundary (G):")
for hypothesis in G:
    print(hypothesis)

print("\nCandidate-Elimination execution completed.")


Training Dataset:
  Sky AirTemp Humidity   Wind Water Forecast EnjoySport
Sunny    Warm   Normal Strong  Warm     Same        Yes
Sunny    Warm     High Strong  Warm     Same        Yes
Rainy    Cold     High Strong  Warm   Change         No
Sunny    Warm     High Strong  Cool   Change        Yes

Example 1: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'] -> Yes
Specific Boundary S: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
General Boundary G: [['?', '?', '?', '?', '?', '?']]

Example 2: ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same'] -> Yes
Specific Boundary S: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
General Boundary G: [['?', '?', '?', '?', '?', '?']]

Example 3: ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'] -> No
Specific Boundary S: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
General Boundary G: [['?', '?', '?', '?', '?', 'Same'], ['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?']]

Example 4: ['Sunny', 'Warm', 'High', 